# A Cardinality-Constrained Portfolio

Find the minimum-risk portfolio that reaches a target expected return while holding **at most k assets**. The risk is a norm, so it is a second-order cone; which assets are held is a set of binary variables. Together they make a mixed-integer conic program, solved here by `Pajarito.jl` with `HiGHS.jl` for the integer part and `Clarabel.jl` for the conic part.

In [ ]:
Pollis.packages("JuMP", "Pajarito", "HiGHS", "Clarabel", "Plots")

In [ ]:
# import Pkg; Pkg.add(["JuMP", "Pajarito", "HiGHS", "Clarabel", "Plots"])
using JuMP, Pajarito, HiGHS, Clarabel, LinearAlgebra, Random, Plots

Random.seed!(1)
n = 12  # assets
F = randn(n, n) ./ 4
Sigma = F' * F + 0.01I  # covariance of the returns
mu = 0.02 .+ 0.08 .* rand(n)  # expected returns
L = cholesky(Symmetric(Sigma)).U  # risk of weights w = norm(L * w)

## The Model

`z[i] = 1` when asset `i` is held, and `w[i] <= z[i]` stops an asset from getting weight unless it is held. `card` and `ret` are kept so their right-hand sides can be changed later.

In [ ]:
pajarito() = optimizer_with_attributes(Pajarito.Optimizer,
    "oa_solver" => optimizer_with_attributes(HiGHS.Optimizer, MOI.Silent() => true),
    "conic_solver" => optimizer_with_attributes(Clarabel.Optimizer, MOI.Silent() => true))

k, rmin = 3, 0.06
model = Model(pajarito())
set_silent(model)
@variable(model, w[1:n] >= 0)
@variable(model, z[1:n], Bin)
@variable(model, risk)
@constraint(model, sum(w) == 1)
@constraint(model, w .<= z)
card = @constraint(model, sum(z) <= k)
ret = @constraint(model, dot(mu, w) >= rmin)
@constraint(model, [risk; L * w] in SecondOrderCone())
@objective(model, Min, risk)
optimize!(model)

held = findall(>(0.5), value.(z))
(status = termination_status(model), risk = objective_value(model), held = held, weights = round.(value.(w)[held]; digits = 3))

## How Much Does the Limit Cost?

Re-solve for each k by changing the right-hand side of `card`. With k = n the limit is not binding, so the last value is the unconstrained minimum risk.

In [ ]:
ks = 1:n
risks = map(ks) do kk
    set_normalized_rhs(card, kk)
    optimize!(model)
    is_solved_and_feasible(model) ? objective_value(model) : NaN
end
set_normalized_rhs(card, k); optimize!(model)  # back to k = 3
plot(ks, risks, marker = :circle, xlabel = "maximum number of assets k", ylabel = "minimum risk", legend = false)

The risk falls quickly for the first few assets and then flattens: once k covers every asset the unconstrained optimum uses, a larger k changes nothing.

## The Continuous Relaxation

Dropping the integrality gives a lower bound. Solve it with Clarabel alone, then give the model Pajarito back before restoring the binaries.

In [ ]:
mip = objective_value(model)
undo = relax_integrality(model)
set_optimizer(model, Clarabel.Optimizer); set_silent(model); optimize!(model)
relaxed = objective_value(model)
set_optimizer(model, pajarito()); set_silent(model); undo(); optimize!(model)
(mixed_integer = mip, relaxation = relaxed, gap = mip - relaxed)

## Next Steps

- Change `rmin` and see which assets enter and leave.
- Replace Clarabel with Hypatia as the `conic_solver` and compare times.